# 00 · Organização do Ambiente

Neste notebook foi configurada a estrutura base do projeto.
Foram definidos o catálogo principal, os três schemas da arquitetura medalhão (bronze, silver e gold)
e o Volume, que é onde os arquivos CSV brutos serão armazenados antes da ingestão.


In [0]:
catalog = "cinedata_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

volume_name = "inputs"
landing_path = f"/Volumes/{catalog}/bronze/{volume_name}"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")


catalog: cinedata_analytics
bronze_schema: cinedata_analytics.bronze
silver_schema: cinedata_analytics.silver
gold_schema: cinedata_analytics.gold
landing_path: /Volumes/cinedata_analytics/bronze/inputs


## Criação do Catálogo, Schemas e Volume


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.{volume_name}")

print("Catalog, schemas e volume prontos.")

Catalog, schemas e volume prontos.


## Validação da Landing Zone

In [0]:
expected_files = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar o volume '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for arq in dbutils.fs.ls(landing_path):
        if arq.name in expected_files:
            tamanho_mb = arq.size / (1024*1024)
            print(f"  - {arq.name} ({tamanho_mb:.2f} MB)")

[OK] Todos os arquivos esperados estão na landing zone:
  - credits_and_tags_IMDB_TMDB.csv (21.29 MB)
  - movies_financials_IMDB_TMDB.csv (1.40 MB)
  - movies_info_TMDB_IMDB.csv (32.35 MB)
  - movies_metrics_IMDB_TMDB.csv (3.17 MB)
  - movies_reviews.csv (1.83 MB)
